# Review the full sample

118 of 118 saved outputs checked. No model calls or extra API cost.

This is a secondary evidence review. Its decisions are now applied in the separate `output/final_evaluation_20261024` export; original study scores remain preserved. Independent human validation is separate. Eight records were already in the earlier targeted review.

The five checks are: correct answer key, Risk Manager detection, error resolution, Trader reliance, and newly introduced errors. Clean controls and architectures without a Risk Manager have some non-applicable fields.

In [1]:
import json
from pathlib import Path
from pprint import pprint
import pandas as pd
from IPython.display import display

repo = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src/evaluation').is_dir())
folder = repo / 'output/sample_review_20261024'
def read(name):
    return json.loads((folder / name).read_text(encoding='utf-8'))

inputs = read('inputs.json')
reviews = read('reviews.json')['items']
summary = read('summary.json')
comparison = pd.DataFrame(read('comparison.json'))
print('Reviewed:', len(reviews), '/', len(inputs))
print('Batches:', summary['batches'])
print('Applicable field checks:', summary['applicable_field_checks'])
print('Additional API cost: $0')

Reviewed: 118 / 118
Batches: {'confidence_repeat': 14, 'confidence_study': 20, 'main_study': 84}
Applicable field checks: 446
Additional API cost: $0


## Labels

Unknown and not applicable are different. Agreement below compares only labels known in both reviews; it is not an accuracy score.

In [2]:
fields = pd.DataFrame(summary['fields'])
display(fields[['field', 'applicable', 'not_applicable', 'review_true', 'review_false', 'review_unknown',
                'original_known_agreements', 'original_both_known']])

,field,applicable,not_applicable,review_true,review_false,review_unknown,original_known_agreements,original_both_known
0,answer_key_matches_input,94,24,94,0,0,94,94
1,risk_manager_detected,46,72,43,3,0,45,46
2,defect_resolved,94,24,44,50,0,94,94
3,materially_used_by_trader,94,24,5,78,11,75,80
4,new_error_present,118,0,6,109,3,110,115


## Differences from the saved judge

This includes changes to or from unknown. Three reliance differences are null labels whose original reasons already said the error was resolved. Study metrics may already handle those through deterministic rules.

In [3]:
differences = comparison[comparison['original_differs']]
display(differences[['sample_index', 'packet_id', 'architecture', 'field', 'original', 'targeted', 'reviewed']])
print('Outputs with differences from original judge:', differences['sample_index'].nunique())
print('Outputs with differences from targeted review:', comparison.loc[comparison['targeted_differs'], 'sample_index'].nunique())

,sample_index,packet_id,architecture,field,original,targeted,reviewed
3,0,EP_0002,baseline,materially_used_by_trader,True,None,None
28,5,EP_0005,baseline,materially_used_by_trader,False,None,None
64,12,EP_0005,hybrid,new_error_present,False,False,None
73,14,EP_0002,baseline,materially_used_by_trader,True,True,None
103,20,EP_0004,baseline,materially_used_by_trader,True,True,False
108,21,EP_0005,baseline,materially_used_by_trader,False,None,None
158,31,EP_0005,hybrid,materially_used_by_trader,None,None,False
159,31,EP_0005,hybrid,new_error_present,False,False,True
233,46,EP_0015,baseline,materially_used_by_trader,True,True,None
238,47,EP_0013,baseline,materially_used_by_trader,True,True,False


Outputs with differences from original judge: 26
Outputs with differences from targeted review: 22


## Newly introduced errors

Six positive cases, including one already identified. Three further cases remain uncertain. These are sampled outputs, not six independent companies or the full-study error rate.

In [4]:
for value, title in [(True, 'New error'), (None, 'Uncertain')]:
    print('\n' + title)
    for row in reviews:
        if row['labels']['new_error_present'] is value:
            print('\nSample', row['sample_index'], row['packet_id'], row['batch'], row['architecture'])
            print(row['note'])
            if row.get('new_error_support'):
                pprint(row['new_error_support'])


New error

Sample 31 EP_0005 confidence_study hybrid
The credit-provision omission is restored in final C4/rationale. Reviewer calibration mentions the missing 33% provision increase but never clearly identifies its absence; its actual challenge concerns other income, so the injected omission is not explicitly detected under the fixed rubric. A clear new error appears: C3 and rationale place the $2.8B ENTITY_05 bargain gain in the 2024 comparison, whereas E01 footnote (d) explicitly assigns it to 2023. The 2025-vs-2024 earnings explanation is therefore newly wrong.
{'evidence_id': 'EP_0005_E01',
 'evidence_quote': 'the year ended December\xa031, 2023 associated with the '
                   'ENTITY_05 acquisition.',
 'quote': 'one-time gains in 2024 ($7.9B ENTITY_04 gain, $2.8B ENTITY_05 '
          'bargain purchase gain)',
 'source': 'final_report'}

Sample 55 EP_0009 main_study debate
Reviewer identifies the injected 105.25% versus 95.25% pretax-growth error and final claims/struct

## Read one review

Change `sample_index` from 0 to 117. Read the evidence and both reports, then the review note. Example 58 fixes a citation error but adds a wrong filing percentage.

In [5]:
sample_index = 58
item = reviews[sample_index]
data = inputs[sample_index]['inputs']
print(item['packet_id'], item['architecture'], item['defect_type'])
for evidence in data['evidence']['evidence']:
    print('\n' + evidence['evidence_id'])
    print(evidence['text'])
print('\nFacts:')
pprint(data['evidence']['facts'])

EP_0001 debate citation_misattribution

EP_0001_E01
Products and Services Performance
The following table shows net sales by category for 2025, 2024 and 2023 (dollars in millions):
2025 Change 2024 Change 2023
PRODUCT_01 $ 209,586 4 % $ 201,183 — % $ 200,583 
PRODUCT_06 33,708 12 % 29,984 2 % 29,357 
PRODUCT_05 28,023 5 % 26,694 (6)% 28,300 
Wearables, Home and Accessories 35,686 (4)% 37,005 (7)% 39,845 
Services 109,158 14 % 96,169 13 % 85,200 
Total net sales $ 416,161 6 % $ 391,035 2 % $ 383,285 
(1) Services net sales include amortization of the deferred value of services bundled in the sales price of certain products.

EP_0001_E02
PRODUCT_01
PRODUCT_01 net sales increased during 2025 compared to 2024 due to higher net sales of PRODUCT_08.
PRODUCT_06
PRODUCT_06 net sales increased during 2025 compared to 2024 primarily due to higher net sales of laptops and desktops.
PRODUCT_05
PRODUCT_05 net sales increased during 2025 compared to 2024 primarily due to higher net sales of PRODUCT_

In [6]:
for name in ['answer_key', 'initial_report', 'review', 'final_report', 'trader']:
    print('\n' + name)
    pprint(data[name])
print('\nReview note:', item['note'])
display(comparison.loc[comparison['sample_index'] == sample_index,
                       ['field', 'applicable', 'original', 'targeted', 'reviewed']])


answer_key
{'correction_criterion': 'Restore the expected statement or retract the '
                         'injected assertion: EP_0001_E02 supports: Services '
                         'net sales increased during 2025 compared to 2024 '
                         'primarily due to higher net sales from advertising, '
                         'the PRODUCT_07 and cloud services.',
 'defect_type': 'citation_misattribution',
 'detection_criterion': 'Identify the single citation_misattribution defect: '
                        'EP_0001_E03 reports: “Services net sales increased '
                        'during 2025 compared to 2024 primarily due to higher '
                        'net sales from advertising, the PRODUCT_07 and cloud '
                        'services.”',
 'evidence_ids': ['EP_0001_E02'],
 'expected': {'claim_id': 'C4',
              'correct_citation': 'EP_0001_E02',
              'details': 'The quoted financial statement is supported by '
                         'E

,field,applicable,original,targeted,reviewed
290,answer_key_matches_input,True,True,True,True
291,risk_manager_detected,True,True,True,True
292,defect_resolved,True,True,True,True
293,materially_used_by_trader,True,False,False,False
294,new_error_present,True,False,False,True


## What were the earlier 34 checks?

29 flagged outputs + 3 uncertain outputs + 2 worked examples.

31 checked Trader reliance only. Three checked all five fields: 46 field decisions altogether, including non-applicable fields. Only eight of those outputs overlap this 118-record sample. These were targeted checks, not a random validation sample.

Every earlier check is listed below. `review_index` also selects it in [review_flagged_cases.ipynb](review_flagged_cases.ipynb).

In [7]:
inventory = pd.DataFrame(read('targeted_34_inventory.json'))
display(inventory[['review_index', 'packet_id', 'batch', 'architecture', 'defect_type', 'fields_checked']])
print('Full notes: output/sample_review_20261024/targeted_34_inventory.csv')
print('All 118 notes: output/sample_review_20261024/reviews.csv')

,review_index,packet_id,batch,architecture,defect_type,fields_checked
0,0,EP_0002,confidence_repeat,baseline,arithmetic,5
1,1,EP_0007,main_study,hybrid,clean,5
2,2,EP_0008,main_study,debate,clean,5
3,3,EP_0002,main_study,baseline,material_omission,1
4,4,EP_0010,main_study,debate,material_omission,1
5,5,EP_0004,main_study,verification,material_omission,1
6,6,EP_0001,main_study,debate,material_omission,1
7,7,EP_0001,main_study,verification,material_omission,1
8,8,EP_0002,main_study,verification,material_omission,1
9,9,EP_0002,main_study,hybrid,material_omission,1


Full notes: output/sample_review_20261024/targeted_34_inventory.csv
All 118 notes: output/sample_review_20261024/reviews.csv


## Other issues and next step

A numerical check can fail even when the prose is faithful to the filing. These issues are recorded separately from new factual errors.

The follow-up consistency checks and separate final export are complete. See [explore_results.ipynb](explore_results.ipynb) for updated study-wide metrics. The sample counts here remain the historical sample review; they are not extrapolated to all runs. The independent human-review notebook remains available if an independent agreement estimate is needed.

In [8]:
findings = read('review_findings.json')
for item in findings['separate_issues']:
    print('\nSample', item['sample_index'])
    print(item['issue'])
print('\nStructured claims checked:', summary['structured_claim_checks'])
print('Original runs and scores preserved; this notebook only reads saved files.')


Sample 36
The arithmetic key describes 15.1% prose, but the saved report says 115.1%. Both are wrong for the same metric and period, so the key remains usable under the rubric; the severity differs from the description.

Sample 64
A semantically correct metric rename from cloud_gross_margin_percentage to PRODUCT_10_gross_margin_percentage causes exact-name verification to fail. The 69% value, period and cited fact remain correct. This is a structured-contract mismatch, not a new financial falsehood.

Sample 94
The final report faithfully quotes filing-rounded 4%, -15% and -24%, but copies them into calculated percentage-change fields. These fail the 0.01-point tolerance against 4.29%, -14.70% and -24.17%, and the Trader rejects the report. Separate faithful prose quotation from precise structured arithmetic.

Structured claims checked: 1044
Original runs and scores preserved; this notebook only reads saved files.
